# Glyph experiment v5k: Qwen2.5-7B × OLMo-2-7B on a free Kaggle GPU

Runs the v5 pipeline from [glyph-alphabet-experiments](https://github.com/devamped1337/glyph-alphabet-experiments) (branch `kaggle`, folder `v5k/`, pinned commit `aa12772906d0`). The steps are determinism, extraction plus J-gram, negotiation, dev, the pre-registered prose gate and depth freeze, test, analysis, and a J-lens comparison.

## How to run
1. **Settings → Accelerator → GPU T4 x2.** The 7B models in fp16 are ~15 GB each and are split across both T4s. A single P100 or T4 (16 GB) is **not enough**, and the first code cell stops if fewer than 2 GPUs are visible.
2. **Settings → Internet → On** (needed for pip, GitHub and Hugging Face). You need a phone-verified Kaggle account to get GPU and Internet.
3. Click **Save Version → Save & Run All (Commit)**. This runs in the background for up to 12 h, and the browser can be closed. Expected time is about 4–6 GPU h.
4. When it finishes, open the version, then **Output**, and download `glyph_v5k_results.zip` (plus `.sha256`). `MANIFEST.txt` inside the zip lists the sha256 of every file.

**Resuming** (if a session stops because of the budget guard, a timeout or an error): open the notebook in the editor, choose **Add Input → Notebook Output**, pick this notebook's last version, and run it again. Cell 3 copies the earlier `glyph_results/` back in, and steps already marked done in `.stages/` are skipped.
Budget: about 12 h per session and 30 GPU h per week. The driver stops before starting a step that would overrun `BUDGET_H` (default 11). Output stays far below 20 GB (about 3 GB). Model weights are cached in `/kaggle/tmp`, which is not saved.

In [ ]:
# 1. environment check + pinned installs (torch is preinstalled on Kaggle)
import subprocess, sys, os, torch
print(sys.version); print("torch", torch.__version__, "cuda", torch.version.cuda)
n = torch.cuda.device_count(); print([torch.cuda.get_device_name(i) for i in range(n)])
assert n >= 2, "Need Accelerator = GPU T4 x2 (a 7B fp16 model does not fit on one 16 GB GPU)."
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.1", "accelerate==1.10.1"], check=True)
REPO_URL, COMMIT = "https://github.com/devamped1337/glyph-alphabet-experiments", "aa12772906d0f5f382d95ce81124790317df63f1"

In [ ]:
# 2. clone the repo at the pinned commit (branch 'kaggle')
REPO = "/kaggle/working/glyph-repo"
if not os.path.exists(REPO): subprocess.run(["git", "clone", "-q", "-b", "kaggle", REPO_URL, REPO], check=True)
subprocess.run(["git", "-C", REPO, "fetch", "-q", "origin", "kaggle"], check=True)
subprocess.run(["git", "-C", REPO, "checkout", "-q", COMMIT], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
import transformers, accelerate; print("transformers", transformers.__version__, "accelerate", accelerate.__version__)

In [ ]:
# 3. paths, env, and resume from a previous version's output (Add Input -> Notebook Output)
import glob, shutil, json
OUT = "/kaggle/working/glyph_results"; os.makedirs(OUT, exist_ok=True)
prev = [d for d in glob.glob("/kaggle/input/*/glyph_results") + glob.glob("/kaggle/input/*/*/glyph_results") if os.path.isdir(os.path.join(d, ".stages"))]
if prev and not os.listdir(os.path.join(OUT)) :
    src = max(prev, key=lambda d: len(os.listdir(os.path.join(d, ".stages")))); print("resuming from", src)
    shutil.copytree(src, OUT, dirs_exist_ok=True)
os.makedirs("/kaggle/tmp/hf", exist_ok=True)
os.environ.update({"GLYPH_OUT": OUT, "GLYPH_HOST": "kaggle", "GLYPH_DEVICE": "cuda", "GLYPH_DTYPE": "float16", "GLYPH_REPO": REPO,
                   "HF_HOME": "/kaggle/tmp/hf", "HF_HUB_ENABLE_HF_TRANSFER": "0", "MAX_RSS_GB": "26", "NTHREADS": "4",
                   "TOKENIZERS_PARALLELISM": "false", "PYTHONUNBUFFERED": "1"})
BUDGET_H = 11.0
done = sorted(os.listdir(os.path.join(OUT, ".stages"))) if os.path.exists(os.path.join(OUT, ".stages")) else []
print("steps already done:", done)
sys.path.insert(0, REPO + "/v5k"); import spec5; print("models:", spec5.MODELS)

## Run the pipeline
One step per process, with checkpoints in `glyph_results/.stages/`. The steps are det_make, det_* (fp16 determinism plus finiteness checks), extract_* and jgram_* (states plus the J-subspace), det_compare, negotiate (CPU, fp32), dev_*, prereg (rule A prose gate ≥ 0.9, rule B depth frozen on dev), test_* (seeds 0,1,2), analyze, and jlens_compare. Re-running this cell is safe.

In [ ]:
# 4. run (or resume) all steps. Steps are skipped if done; the budget guard stops before overrunning the session
import time; T0 = time.time()
rc = subprocess.run([sys.executable, REPO + "/v5k/run_kaggle5k.py", "all", "--budget-h", str(BUDGET_H)]).returncode
print("driver exit", rc, f"{(time.time()-T0)/3600:.2f} h")
for f in sorted(glob.glob(OUT + "/.stages/*.done")): print(json.load(open(f)))

In [ ]:
# 5. results: frozen settings, tables, J-lens comparison
for n in ("frozen5.json", "det_report.json", "jlens_compare5k.json"):
    p = os.path.join(OUT, n)
    if os.path.exists(p): print("=====", n); print(json.dumps(json.load(open(p)), indent=1, default=str)[:3000])
p = os.path.join(OUT, "TABLES_v5.md")
if os.path.exists(p):
    from IPython.display import Markdown, display; display(Markdown(open(p).read()))

In [ ]:
# 6. package: MANIFEST.txt (sha256 of every file) + zip for download (written even if incomplete, so partial runs can be inspected)
subprocess.run([sys.executable, REPO + "/v5k/package5k.py"], check=True)
print(open("/kaggle/working/glyph_v5k_results.zip.sha256").read())
# remove the clone so the saved output holds only results (it is re-cloned on resume)
shutil.rmtree(REPO, ignore_errors=True)
assert rc == 0, "pipeline incomplete: see driver message above; resume by adding this version's output as input and re-running"